<a href="https://colab.research.google.com/github/rontalapoojareddy/DeepLearning1/blob/main/Assignment7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**2503B05130**

**R.Pooja Reddy**

**M.Tech CSE**

**Install Required Libraries**

In [ ]:
# =====================================================
# STEP 1: Install Required Libraries
# =====================================================
!pip install transformers datasets evaluate seqeval accelerate -q


**Imports**

In [ ]:
# =====================================================
# STEP 2: Imports
# =====================================================
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    DataCollatorForLanguageModeling,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments
)


**Load Tokenizer (Base Model)**

In [ ]:
# =====================================================
# STEP 3: Load Tokenizer (Base Model)
# =====================================================
model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

**Self-Supervised Pretraining (MLM)**

In [ ]:
# =====================================================
# STEP 4: Self-Supervised Pretraining (MLM)
# =====================================================

print("Loading Unlabeled Dataset for MLM...")

# Using WikiText small version for demo
unlabeled_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_mlm(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_unlabeled = unlabeled_dataset.map(
    tokenize_mlm,
    batched=True,
    remove_columns=["text"]
)

data_collator_mlm = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

mlm_model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

training_args_mlm = TrainingArguments(
    output_dir="./mlm_model",
    per_device_train_batch_size=16,
    num_train_epochs=1,
    logging_steps=100,
    save_strategy="no"
)

trainer_mlm = Trainer(
    model=mlm_model,
    args=training_args_mlm,
    train_dataset=tokenized_unlabeled["train"],
    data_collator=data_collator_mlm
)

print("Starting MLM Pretraining...")
trainer_mlm.train()

mlm_model.save_pretrained("./mlm_model")
tokenizer.save_pretrained("./mlm_model")

Loading Unlabeled Dataset for MLM...


Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting MLM Pretraining...


Step,Training Loss
100,2.137263


Step,Training Loss
100,2.137263
200,1.998987
300,1.980225
400,1.923849
500,1.949998
600,1.938127
700,1.879792
800,1.921899
900,1.931596
1000,1.879770


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mlm_model/tokenizer_config.json', './mlm_model/tokenizer.json')

**Load CoNLL-2003 Manually (No HF Script)**

In [ ]:
# =====================================================
# STEP 5: Load CoNLL-2003 Manually (No HF Script)
# =====================================================

import requests
from datasets import Dataset, DatasetDict

def load_conll_file(url):
    response = requests.get(url)
    # Add a check for empty response content
    if not response.text.strip():
        print(f"Warning: No content received from {url}")
        return [], []

    lines = response.text.split("\n")

    tokens = []
    ner_tags = []

    all_tokens = []
    all_tags = []

    for line in lines:
        if line.strip() == "":
            if tokens: # Only append if there are tokens
                all_tokens.append(tokens)
                all_tags.append(ner_tags)
                tokens = []
                ner_tags = []
        else:
            parts = line.split()
            if len(parts) > 1: # Ensure there's at least a token and a tag
                tokens.append(parts[0])
                ner_tags.append(parts[-1])
            else:
                # Optionally print a warning for malformed lines if needed
                pass

    # Append any remaining tokens/tags after the loop
    if tokens:
        all_tokens.append(tokens)
        all_tags.append(ner_tags)

    return all_tokens, all_tags


print("Downloading CoNLL-2003...")

train_tokens, train_tags = load_conll_file(
    "https://raw.githubusercontent.com/chakki-works/seqeval/master/example/conll2003/train.txt"
)

valid_tokens, valid_tags = load_conll_file(
    "https://raw.githubusercontent.com/chakki-works/seqeval/master/example/conll2003/dev.txt"
)

test_tokens, test_tags = load_conll_file(
    "https://raw.githubusercontent.com/chakki-works/seqeval/master/example/conll2003/test.txt"
)

# Create label mapping
unique_labels = sorted(list(set(tag for sentence in train_tags for tag in sentence)))
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}
num_labels = len(unique_labels)
# Create label_list for evaluation metric (ensure it's ordered by ID)
label_list = [id_to_label[i] for i in range(num_labels)]

print("Labels:", unique_labels)
print("Number of labels:", num_labels)
print("Label list for evaluation:", label_list)

# Create Dataset objects for each split
train_dataset = Dataset.from_dict({"tokens": train_tokens, "tags": train_tags})
valid_dataset = Dataset.from_dict({"tokens": valid_tokens, "tags": valid_tags})
test_dataset = Dataset.from_dict({"tokens": test_tokens, "tags": test_tags})

# Combine into a DatasetDict
ner_dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset
})

print("CoNLL-2003 Dataset Loaded:")
print(ner_dataset)

Labels: ['Found']
Number of labels: 1
Label list for evaluation: ['Found']
CoNLL-2003 Dataset Loaded:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1
    })
})


**Load Token Classification Model**

In [ ]:
# =====================================================
# STEP 7: Load Token Classification Model
# =====================================================

ner_model = AutoModelForTokenClassification.from_pretrained(
    "./mlm_model",  # Use pretrained MLM weights
    num_labels=num_labels
)


/usr/local/lib/python3.12/dist-packages/torch/utils/_device.py:109: UserWarning: Initializing zero-element tensors is a no-op
  return func(*args, **kwargs)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ./mlm_model
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Evaluation Metric**

In [ ]:
# =====================================================
# STEP 8: Evaluation Metric
# =====================================================

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


**Tokenize & Align Labels (Corrected)**

In [ ]:
# =====================================================
# STEP 6: Tokenize & Align Labels (Corrected)
# =====================================================

# The tokenizer was already loaded in Step 3 (QhjFcEfoqMNJ).
# The 'from transformers import AutoTokenizer' and
# 'tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")' lines are removed
# to use the globally defined 'tokenizer' and avoid inconsistencies.

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    for i, tags in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Map string label to its ID using the globally defined label_to_id
                label_ids.append(label_to_id[tags[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


print("Verifying ner_dataset before tokenization:")
print(ner_dataset)

# Apply tokenization to the DatasetDict
tokenized_ner = ner_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

# Remove the original 'tokens' and 'tags' columns as they are no longer needed for training
tokenized_ner = tokenized_ner.remove_columns(["tokens", "tags"])

print("Tokenized NER Dataset:")
print(tokenized_ner)

Verifying ner_dataset before tokenization:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1
    })
})


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenized NER Dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})
